# Week 5: Forecasting spending

Goal: two forecasts per expense category, served by `POST /forecast` and shown in the app:
- **End of month:** *"At this pace, Food & Drinks will reach about ₱8,200 by Oct 31."*
- **Next month:** *"November: about ₱7,900 on Food & Drinks."*

You'll build features with no look-ahead, split by **time**, and check whether gradient boosting beats a simple baseline.

Spec: `docs/superpowers/specs/2026-10-23-spending-forecast-design.md`. Generate the data first: `.venv\Scripts\python scripts\generate_fake_data.py --years 3`

## Setup
The 3-year file has growth (~5% a year), a December bump and busier weekends, so there is something to learn. The dates are treated as the phone's local dates.

`CATEGORIES` is fixed: a category's position is its `category_code`, and the service uses the same list. The feature lists are the exact columns the service builds.

In [ ]:
import calendar
import json
from datetime import date
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor

CATEGORIES = ['Food & Drinks', 'Coffee & Snacks', 'Groceries', 'Transportation', 'Bills & Utilities',
              'Shopping', 'Healthcare', 'Entertainment', 'other']
EOM_FEATURES = ['category_code', 'day', 'days_left', 'weekend_days_left', 'spent_ratio',
                'last_month_ratio', 'avg6_ratio', 'last_month_share_by_day', 'month_of_year']
NEXT_FEATURES = ['category_code', 'last_month_ratio', 'avg6_ratio', 'month_of_year']

df = pd.read_csv('../data/fake_transactions_3y.csv')
exp = df[df['Type'] == 'expense'].copy()
exp['day'] = pd.to_datetime(exp['Date']).dt.tz_localize(None).dt.normalize()

# Use whole months only: the file starts and ends mid-month
FIRST_DAY = exp['day'].min() + pd.offsets.MonthBegin(1)
LAST_FULL_MONTH = exp['day'].max().to_period('M').to_timestamp() - pd.offsets.MonthBegin(1)
LAST_DAY = LAST_FULL_MONTH + pd.offsets.MonthEnd(0)
exp = exp[(exp['day'] >= FIRST_DAY) & (exp['day'] <= LAST_DAY)]
print(FIRST_DAY.date(), '→', LAST_DAY.date(), len(exp), 'expenses')

---
## A. Daily table
Build three tables, one column per category:
- `daily`: total spent per **day**. Hint: `pivot_table(index='day', columns='Category', values='Amount', aggfunc='sum', fill_value=0)`, then `reindex(pd.date_range(FIRST_DAY, LAST_DAY, freq='D'), fill_value=0)` so days with no spending are 0
- `monthly`: total per month. Hint: `daily.resample('MS').sum()` (index = first day of each month)
- `cum`: running total **within** each month. Hint: `daily.groupby(daily.index.to_period('M')).cumsum()`

In [ ]:
# TODO


---
## B. Features (no look-ahead)
### B1. End-of-month rows
One row per **(category, month M, cut-off day d = 1…28)**. "Full months" are the months before M, starting at the category's first month with spending. Skip M if it has fewer than **6** full months before it, if `avg3 <= 0`, or if M is after `LAST_FULL_MONTH`.

| column | meaning |
|---|---|
| `category`, `month`, `spent`, `avg3`, `actual` | kept for scoring (`actual` = total of month M) |
| `category_code` | `CATEGORIES.index(category)` |
| `day`, `days_left` | d, and days in M minus d |
| `weekend_days_left` | Saturdays + Sundays from day d+1 to the end of M |
| `spent_ratio` | spent on days 1…d of M, ÷ `avg3` |
| `last_month_ratio`, `avg6_ratio` | last full month's total ÷ `avg3`; mean of the last 6 full months ÷ `avg3` |
| `last_month_share_by_day` | share of last month's total spent by the same day last month (use day `min(d, days in last month)`); 0 if last month was 0 |
| `month_of_year` | M's month number |
| `eom_ratio` (**target**) | (`actual` − `spent`) ÷ `avg3`, the rest of the month in "usual months" |

Hints: loop over `daily.columns` and `monthly.index`. `hist = tot[(tot.index >= first) & (tot.index < M)]` gives the full months. `cum.loc[M + pd.Timedelta(days=d - 1), cat]` is spent through day d. `M.days_in_month` and `date(y, m, k).weekday() >= 5` help with the calendar. Build a list of dicts, then `eom = pd.DataFrame(rows)`.

**Think:** why divide everything by `avg3` instead of predicting pesos? (Hint: will a real user spend like the fake one?)

In [ ]:
# TODO


### B2. Next-month rows
One row per **(category, month M)**: "we are somewhere in M, predict M+1". Only **full months before M** are used, since M itself is unfinished.

Columns: `category`, `month` (= M), `avg3`, `actual` (= total of M+1), `category_code`, `last_month_ratio`, `avg6_ratio`, `month_of_year` (of **M+1**), and the target `next_ratio` = total(M+1) ÷ `avg3`. Same skip rules; also skip when M+1 is after `LAST_FULL_MONTH`. Save as `nxt`.

In [ ]:
# TODO


These checks recompute one row of each table straight from the raw expenses. If they pass, there is no look-ahead.

In [ ]:
def month_total(cat, start):
    m = exp[(exp['Category'] == cat) & (exp['day'] >= start) & (exp['day'] < start + pd.offsets.MonthBegin(1))]
    return m['Amount'].sum()

r = eom[(eom['day'] == 10)].iloc[len(eom[eom['day'] == 10]) // 2]
past = [month_total(r['category'], r['month'] - pd.offsets.MonthBegin(k)) for k in (1, 2, 3)]
spent = exp[(exp['Category'] == r['category']) & (exp['day'] >= r['month']) & (exp['day'] <= r['month'] + pd.Timedelta(days=9))]['Amount'].sum()
assert abs(r['avg3'] - np.mean(past)) < 1e-6 and abs(r['spent_ratio'] - spent / np.mean(past)) < 1e-9
assert abs(r['eom_ratio'] - (month_total(r['category'], r['month']) - spent) / np.mean(past)) < 1e-9

n = nxt.iloc[len(nxt) // 2]
past = [month_total(n['category'], n['month'] - pd.offsets.MonthBegin(k)) for k in (1, 2, 3)]
assert abs(n['last_month_ratio'] - past[0] / np.mean(past)) < 1e-9
assert abs(n['next_ratio'] - month_total(n['category'], n['month'] + pd.offsets.MonthBegin(1)) / np.mean(past)) < 1e-9
assert list(eom[EOM_FEATURES].columns) == EOM_FEATURES and list(nxt[NEXT_FEATURES].columns) == NEXT_FEATURES
print('no look-ahead ✓')

---
## C. Split by time
Test = the **last 6 full months**, train = everything before. For `nxt`, a row belongs to the month it **predicts** (M+1).

Make `eom_train`, `eom_test`, `nxt_train`, `nxt_test`. Hint: `test_from = LAST_FULL_MONTH - pd.offsets.MonthBegin(5)`.

**Think:** why would `train_test_split(..., shuffle=True)` give a score that's too good? (Hint: rows from the same month share almost everything.)

In [ ]:
# TODO


---
## D. End-of-month model
Scoring helper (given): turns a predicted ratio back into pesos and returns the mean absolute error.

In [ ]:
def pesos(pred_ratio, rows):
    """Predicted month total in pesos (end of month) or next month's total."""
    spent = rows['spent'] if 'spent' in rows else 0
    return spent + np.clip(pred_ratio, 0, None) * rows['avg3']


def mae_pesos(pred_ratio, rows):
    return float((pesos(pred_ratio, rows) - rows['actual']).abs().mean())

Compare three predictions of `eom_ratio` on `eom_test`, and save them in a dict `eom_preds`:
- **baseline:** the pace so far: `spent / day × days_in_month`, turned into a ratio: `(that − spent) / avg3`
- **HistGB:** `HistGradientBoostingRegressor(random_state=0)` fitted on `eom_train[EOM_FEATURES]` → `eom_ratio`. Save the model as `hgb_eom`
- **LightGBM:** `lgb.LGBMRegressor(random_state=0, verbose=-1)`, same data. Save as `lgb_eom`

Print `mae_pesos(...)` for each.

In [ ]:
# TODO


### D2. Error by cut-off day
Show each method's MAE grouped by `day` (rows 1, 3, 5, 7, 10, 15, 20, 25, 28). Hint: `(pesos(pred, eom_test) - eom_test['actual']).abs().groupby(eom_test['day']).mean()`.

**Think:** Ollie warns "on pace to go over budget" from **day 5**. Is the error at day 5 small enough, or should it wait longer? And which method wins near the **end** of the month? Why might that be?

In [ ]:
# TODO


---
## E. Next-month model
Same comparison on `nxt_test`, saved as `next_preds`:
- **baseline:** `avg3`, which is a ratio of exactly 1
- **HistGB** (`hgb_next`) and **LightGBM** (`lgb_next`; try `min_child_samples=5`, there are few rows) on `NEXT_FEATURES` → `next_ratio`

**Think:** how many training rows does this model have, compared with D? What does that do to boosting?

In [ ]:
# TODO


---
## F. Decide
**Gate:** a model ships only if its test MAE is **lower than the baseline's**. Otherwise the baseline ships.

Set `EOM_CHOICE` and `NEXT_CHOICE` to `'HistGB'`, `'LightGBM'` or `'baseline'`, and put your fitted models in `FITTED` (`{'eom': {'HistGB': hgb_eom, 'LightGBM': lgb_eom}, 'next': {...}}`). Then run the cells below.

LightGBM may not load on Vercel. If it wins by only a little, HistGB is the safer pick.

In [ ]:
# TODO
EOM_CHOICE = ...
NEXT_CHOICE = ...
FITTED = ...


In [ ]:
# Gate (given)
for name, preds, choice in [('end of month', eom_preds, EOM_CHOICE), ('next month', next_preds, NEXT_CHOICE)]:
    scores = {k: mae_pesos(v, eom_test if name == 'end of month' else nxt_test) for k, v in preds.items()}
    ok = choice == 'baseline' or scores[choice] < scores['baseline']
    print(f"{name}: ships {choice} (MAE ₱{scores[choice]:,.0f} vs baseline ₱{scores['baseline']:,.0f})", '✓' if ok else '✗ fails the gate')
    assert ok

In [ ]:
# Export (given): models for the service, plus a sample the service test compares its features against
def bundle(choice, kind, train, features, target):
    if choice == 'baseline':
        return {'model': None, 'features': features}
    model = clone(FITTED[kind][choice]).fit(train[features].to_numpy(), train[target].to_numpy())  # NumPy in, so the service needs no pandas
    return {'model': model, 'features': features}

service = Path('../service')
joblib.dump(bundle(EOM_CHOICE, 'eom', eom_train, EOM_FEATURES, 'eom_ratio'), service / 'model' / 'forecast_eom.joblib')
joblib.dump(bundle(NEXT_CHOICE, 'next', nxt_train, NEXT_FEATURES, 'next_ratio'), service / 'model' / 'forecast_next.joblib')

TODAY = pd.Timestamp('2026-08-15')
month = TODAY.to_period('M').to_timestamp()
start = month - pd.offsets.MonthBegin(6)
sample = {'request': {'today': str(TODAY.date()), 'series': []}, 'eom': {}, 'next': {}}
for key, cat in [('food', 'Food & Drinks'), ('bills', 'Bills & Utilities')]:
    days = daily.loc[start:TODAY, cat]
    sample['request']['series'].append({'key': key, 'category': cat,
                                        'daily': {str(d.date()): round(float(v), 2) for d, v in days.items() if v > 0}})
    e = eom[(eom['category'] == cat) & (eom['month'] == month) & (eom['day'] == TODAY.day)]
    n = nxt[(nxt['category'] == cat) & (nxt['month'] == month)]
    sample['eom'][key] = {k: float(v) for k, v in e[EOM_FEATURES].iloc[0].items()}
    sample['next'][key] = {k: float(v) for k, v in n[NEXT_FEATURES].iloc[0].items()}
(service / 'tests' / 'fixtures').mkdir(exist_ok=True)
(service / 'tests' / 'fixtures' / 'forecast_sample.json').write_text(json.dumps(sample, indent=1))
print('saved models and sample ✓')

**Note:** the sample uses amounts rounded to centavos, the same as the app sends, so its features can differ from the table in the 10th decimal. The service test allows for that.

**My choices:** TODO (end of month: …, next month: …)

**Why:** TODO, with the MAE numbers. Also: is day 5 safe for Ollie's pace warning? (D2)